### 04 — Clinical Feature Extraction using ClinicalBERT

What this notebook does:

- load the existing training, validation, and test splits
- inspect the clinical variables used by the model
- convert the structured clinical data into short natural-language text
- load the pretrained `Bio_ClinicalBERT` tokenizer and model
- test the tokenisation process
- extract contextual clinical token embeddings
- use the reusable `ClinicalBERTEncoder` class from `src/clinical_encoder.py`
- freeze the pretrained ClinicalBERT parameters
- test ClinicalBERT on a batch of clinical text
- save the generated `clinical_text` column back into the dataset splits.

The output from this notebook is a sequence of 768-dimensional clinical token features that will later be combined with the EfficientNet visual features during multimodal fusion.


In [ ]:
from pathlib import Path

import pandas as pd
import torch
import torch.nn as nn

from transformers import AutoTokenizer, AutoModel

### 1. Define Project Paths

The notebook uses the existing project folders for the raw ultrasound images, processed dataset splits, and trained model checkpoints.

In [ ]:
PROJECT_ROOT = Path("..").resolve()

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"

train_df = pd.read_csv(PROCESSED_DIR / "train.csv")
val_df = pd.read_csv(PROCESSED_DIR / "val.csv")
test_df = pd.read_csv(PROCESSED_DIR / "test.csv")

print(len(train_df), len(val_df), len(test_df))

### 2. Inspect the Clinical Data

The clinical branch uses four structured variables:

- Age
- Tissue composition
- Signs
- Symptoms

These variables describe patient-level clinical information and are used as the input to ClinicalBERT.


In [ ]:
clinical_columns = [
    "Age",
    "Tissue_composition",
    "Signs",
    "Symptoms"
]

display(train_df[clinical_columns].head(10))

Inspect the value counts for each variable and identify entries such as 'not available' that are stored as text in the dataset.


In [ ]:
for column in clinical_columns:
    print(f"\n--- {column} ---")
    print(train_df[column].value_counts(dropna=False).head(20))

### 3. Convert Structured Clinical Data into Text

ClinicalBERT expects text rather than separate spreadsheet columns.

The four clinical variables are combined into a short sentence using a template

Using the same format for every case gives ClinicalBERT a consistent representation of the structured clinical information.


In [ ]:
def create_clinical_text(row):
    age = str(row["Age"]).strip()
    tissue = str(row["Tissue_composition"]).strip()
    signs = str(row["Signs"]).strip()
    symptoms = str(row["Symptoms"]).strip()

    text = (
        f"Patient age: {age} years. "
        f"Tissue composition: {tissue}. "
        f"Clinical signs: {signs}. "
        f"Symptoms: {symptoms}."
    )

    return text

The text representation is created for the training, validation, and test sets and stored in a new clinical_text column.


In [ ]:
train_df["clinical_text"] = train_df.apply(
    create_clinical_text,
    axis=1
)

val_df["clinical_text"] = val_df.apply(
    create_clinical_text,
    axis=1
)

test_df["clinical_text"] = test_df.apply(
    create_clinical_text,
    axis=1
)

In [ ]:
for text in train_df["clinical_text"].head(5):
    print(text)
    print()

### 4. Load ClinicalBERT

The clinical text encoder uses `emilyalsentzer/Bio_ClinicalBERT`.

Both the pretrained tokenizer and pretrained model are loaded from Hugging Face.

ClinicalBERT produces contextual token representations that can capture relationships between the different clinical terms in the input text.


In [ ]:
MODEL_NAME = "emilyalsentzer/Bio_ClinicalBERT"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

clinical_bert = AutoModel.from_pretrained(
    MODEL_NAME
)

The model is moved to the GPU when CUDA is available; otherwise, it runs on the CPU.


In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

clinical_bert = clinical_bert.to(device)

print("Device:", device)

### 5. Test the Tokenisation Process

The tokenizer converts the text into model inputs including:

- `input_ids` — token identifiers
- `attention_mask` — indicates which positions contain real tokens rather than padding.

In [ ]:
sample_text = train_df.iloc[0]["clinical_text"]

print(sample_text)

In [ ]:
encoded = tokenizer(
    sample_text,
    return_tensors="pt",
    padding=True,
    truncation=True,
    max_length=64
)

print(encoded.keys())
print(encoded["input_ids"].shape)
print(encoded["attention_mask"].shape)

### 6. Extract ClinicalBERT Features

The tokenised inputs are moved to the selected device and passed through ClinicalBERT in evaluation mode.

torch.no_grad() is used because this step is only testing feature extraction and does not require gradient calculations.


In [ ]:
encoded = {
    key: value.to(device)
    for key, value in encoded.items()
}

In [ ]:
clinical_bert.eval()

with torch.no_grad():
    output = clinical_bert(**encoded)

In [ ]:
print(output.last_hidden_state.shape)

### 7. Use the Reusable ClinicalBERT Encoder

The `ClinicalBERTEncoder` class from `src/clinical_encoder.py` is imported.


In [ ]:
import sys

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.clinical_encoder import ClinicalBERTEncoder

In [ ]:
clinical_encoder = ClinicalBERTEncoder().to(device)

### 8. Freeze the ClinicalBERT Parameters

The pretrained ClinicalBERT weights are frozen before multimodal training.


In [ ]:
for parameter in clinical_encoder.bert.parameters():
    parameter.requires_grad = False

In [ ]:
print(
    sum(
        p.numel()
        for p in clinical_encoder.parameters()
        if p.requires_grad
    )
)

### 9. Test ClinicalBERT on a Batch

A batch of eight clinical-text samples is tokenised together.

In [ ]:
texts = train_df["clinical_text"].iloc[:8].tolist()

batch = tokenizer(
    texts,
    padding=True,
    truncation=True,
    max_length=64,
    return_tensors="pt"
)

The tokenised batch is moved to the same device as the ClinicalBERT encoder.


In [ ]:
batch = {
    key: value.to(device)
    for key, value in batch.items()
}

The batch is passed through the clinical encoder.

The resulting tensor contains one 768-dimensional feature vector for every token in every clinical-text sequence.


In [ ]:
clinical_encoder.eval()

with torch.no_grad():
    clinical_tokens = clinical_encoder(
        input_ids=batch["input_ids"],
        attention_mask=batch["attention_mask"],
        token_type_ids=batch.get("token_type_ids")
    )

print(clinical_tokens.shape)

### 10. Save the Clinical Text into the Dataset Splits

The updated training, validation, and test DataFrames are saved back to the processed CSV files.


In [ ]:
train_df.to_csv(
    PROCESSED_DIR / "train.csv",
    index=False
)

val_df.to_csv(
    PROCESSED_DIR / "val.csv",
    index=False
)

test_df.to_csv(
    PROCESSED_DIR / "test.csv",
    index=False
)